In [1]:
# =============================================================================
# NB10 — Centralized Single-Model Run (Upper Bound Reference)
#
# ┌─────────────────────────────────────────────────────────────────────┐
# │  CHANGE THIS ONE LINE TO SWITCH MODELS:                               │
# │                                                                       │
# │  MODEL_NAME = "efficientnet_b0"       ~45 min  (baseline)             │
# │  MODEL_NAME = "mobilenetv3_large_100" ~30 min  (fastest)              │
# │  MODEL_NAME = "densenet121"           ~50 min  (medical SOTA)         │
# │  MODEL_NAME = "efficientnet_b3"       ~60 min  (larger capacity)      │
# │  MODEL_NAME = "resnet50"              ~60 min  (classic)              │
# └─────────────────────────────────────────────────────────────────────┘
#
# Run one notebook per model. Each saves its own .pkl to:
#   /kaggle/working/results/centralized_multimodel/results_{model_name}.pkl
#
# After all 5 runs, NB8 loads all .pkl files to compute federation cost:
#   federation_cost = centralized_AUC − fedadapt_ef_AUC  (per backbone)
# =============================================================================

# ─── ONLY LINE YOU NEED TO CHANGE ────────────────────────────────────────────
MODEL_NAME = "densenet121"
# ─────────────────────────────────────────────────────────────────────────────


# =============================================================================
# SECTION 0 — INSTALL & IMPORTS
# =============================================================================

import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import timm
    print(f"timm {timm.__version__} already installed")
except ImportError:
    print("Installing timm...")
    pip_install("timm>=0.9.0")
    import timm

import os, gc, time, copy, pickle, json, warnings, random
from PIL import Image
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

try:
    from torch.amp import autocast, GradScaler
    _AMP_NEW = True
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    _AMP_NEW = False

import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             precision_score, recall_score, roc_curve)
from tqdm.auto import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("✅ All imports done")
print(f"PyTorch: {torch.__version__} | timm: {timm.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


# =============================================================================
# SECTION 1 — MODEL REGISTRY  (identical to NB9)
# =============================================================================

MODEL_REGISTRY = {
    "efficientnet_b0": {
        "timm_name":   "efficientnet_b0",
        "batch_size":  128,
        "backbone_lr": 3e-5,
        "head_lr":     3e-4,
        "description": "EfficientNet-B0 (5.3M) — Current baseline",
        "color":       "#E74C3C",
    },
    "efficientnet_b3": {
        "timm_name":   "efficientnet_b3",
        "batch_size":  96,
        "backbone_lr": 2e-5,
        "head_lr":     2e-4,
        "description": "EfficientNet-B3 (12M) — Larger capacity",
        "color":       "#E67E22",
    },
    "densenet121": {
        "timm_name":   "densenet121",
        "batch_size":  96,
        "backbone_lr": 3e-5,
        "head_lr":     3e-4,
        "description": "DenseNet-121 (8M) — Medical imaging standard",
        "color":       "#2ECC71",
    },
    "mobilenetv3_large_100": {
        "timm_name":   "mobilenetv3_large_100",
        "batch_size":  128,
        "backbone_lr": 5e-5,
        "head_lr":     5e-4,
        "description": "MobileNetV3-Large (5.5M) — Lightest / fastest",
        "color":       "#3498DB",
    },
    "resnet50": {
        "timm_name":   "resnet50",
        "batch_size":  96,
        "backbone_lr": 2e-5,
        "head_lr":     2e-4,
        "description": "ResNet-50 (25M) — Classic baseline",
        "color":       "#9B59B6",
    },
}

# Validate MODEL_NAME early
assert MODEL_NAME in MODEL_REGISTRY, \
    f"Unknown model '{MODEL_NAME}'. Choose from: {list(MODEL_REGISTRY.keys())}"

print(f"\n📋 Running: {MODEL_NAME}")
print(f"   {MODEL_REGISTRY[MODEL_NAME]['description']}")
print(f"   batch_size={MODEL_REGISTRY[MODEL_NAME]['batch_size']}")


# =============================================================================
# SECTION 2 — CONFIGURATION
# =============================================================================

@dataclass
class Config:
    # ── Paths ─────────────────────────────────────────────────────────────────
    UNIFIED_CSV: str = "/kaggle/input/datasets/ariroyjit/unified-dr-dataset/unified_dataset.csv"
    STATS_JSON:  str = "/kaggle/input/datasets/ariroyjit/unified-dr-dataset/dataset_stats.json"
    SAVE_DIR:    str = "/kaggle/working/results/centralized_multimodel"

    # ── Model (set from MODEL_NAME above) ─────────────────────────────────────
    MODEL_NAME:  str   = MODEL_NAME
    PRETRAINED:  bool  = True
    DROPOUT:     float = 0.4

    # ── Data splits — MUST match NB9 exactly ──────────────────────────────────
    TRAIN_RATIO: float = 0.70
    VAL_RATIO:   float = 0.15
    TEST_RATIO:  float = 0.15

    # ── Training ──────────────────────────────────────────────────────────────
    MAX_EPOCHS:    int   = 30
    FREEZE_EPOCHS: int   = 3       # Freeze backbone for first N epochs
    FREEZE_BN:     bool  = True    # Keep BN frozen throughout
    WEIGHT_DECAY:  float = 1e-4
    GRAD_CLIP:     float = 1.0
    POS_WEIGHT:    float = 2.0     # Same clinical override as NB7/NB9

    # ── LRs — set per model from MODEL_REGISTRY ───────────────────────────────
    BACKBONE_LR: float = MODEL_REGISTRY[MODEL_NAME]["backbone_lr"]
    HEAD_LR:     float = MODEL_REGISTRY[MODEL_NAME]["head_lr"]
    BATCH_SIZE:  int   = MODEL_REGISTRY[MODEL_NAME]["batch_size"]

    # ── Early stopping ────────────────────────────────────────────────────────
    PATIENCE:  int   = 5
    MIN_DELTA: float = 0.001

    # ── Seeds — identical to NB9 for fair comparison ──────────────────────────
    SEEDS: List[int] = field(default_factory=lambda: [42, 123])

    # ── Hardware ──────────────────────────────────────────────────────────────
    DEVICE:      str  = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS: int  = 4
    PIN_MEMORY:  bool = True
    USE_AMP:     bool = True

config = Config()

# Auto-find CSV (handles different Kaggle mount slug variations)
for _cp in [
    config.UNIFIED_CSV,
    "/kaggle/input/unified-dr-dataset/unified_dataset.csv",
    "/kaggle/input/datasets/ariroyjit/unified-dr-dataset/unified_dataset.csv",
]:
    if os.path.exists(_cp):
        config.UNIFIED_CSV = _cp
        break

# Per-model save dir
MODEL_SAVE_DIR = Path(config.SAVE_DIR) / MODEL_NAME
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✅ Config ready")
print(f"   Model:      {config.MODEL_NAME}")
print(f"   Seeds:      {config.SEEDS}")
print(f"   Max epochs: {config.MAX_EPOCHS} (early stop patience={config.PATIENCE})")
print(f"   Batch size: {config.BATCH_SIZE}")
print(f"   Save dir:   {MODEL_SAVE_DIR}")


# =============================================================================
# SECTION 3 — UTILITIES
# =============================================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def worker_init_fn(worker_id):
    np.random.seed(torch.initial_seed() % 2**32 + worker_id)


# =============================================================================
# SECTION 4 — DATASET  (identical splits to NB9)
# =============================================================================

def get_transforms(split='train'):
    try:
        with open(config.STATS_JSON) as f:
            st = json.load(f)
        mean, std = st['mean'], st['std']
    except Exception:
        mean = [0.485, 0.456, 0.406]
        std  = [0.229, 0.224, 0.225]

    norm = transforms.Normalize(mean, std)

    if split == 'train':
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.ToTensor(),
            norm,
        ])
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        norm,
    ])


class DRDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['image_path']).convert('RGB')
        except FileNotFoundError:
            img = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(float(row['binary_label']), dtype=torch.float32)
        return img, label


def remap_image_paths(df):
    """Auto-fix stale absolute paths from NB0 by scanning /kaggle/input/."""
    sample  = df['image_path'].iloc[:20].tolist()
    n_valid = sum(1 for p in sample if os.path.exists(p))
    if n_valid >= 18:
        print(f"  ✅ Paths OK ({n_valid}/20 sampled exist) — no remapping needed")
        return df

    print(f"  ⚠️  Broken paths ({n_valid}/20 exist) — scanning /kaggle/input/ ...")
    filename_to_real = {}
    total = 0
    for slug in sorted(os.listdir('/kaggle/input')):
        slug_dir = f'/kaggle/input/{slug}'
        count = 0
        for root, _, files in os.walk(slug_dir):
            for f in files:
                if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    if f not in filename_to_real:
                        filename_to_real[f] = os.path.join(root, f)
                    count += 1
                    total += 1
        print(f"    [{slug}]: {count:,} images indexed")

    print(f"  📊 Total indexed: {total:,} images")
    df = df.copy()
    df['image_path'] = df['image_path'].apply(
        lambda p: filename_to_real.get(os.path.basename(p), p)
                  if not os.path.exists(p) else p
    )
    n_fixed = sum(1 for p in df['image_path'].iloc[:20] if os.path.exists(p))
    print(f"  ✅ After remap: {n_fixed}/20 sampled paths valid")
    if n_fixed < 15:
        print("  ❌ Still broken — make sure ALL source datasets are attached:")
        print("     • APTOS 2019 Blindness Detection")
        print("     • DDR dataset")
        print("     • IDRiD / Indian Diabetic Retinopathy")
        print("     • unified-dr-dataset")
    return df


def load_and_split_data():
    """
    Load with IDENTICAL splits to NB9.
    random_state=42 everywhere → same train/val/test rows across all notebooks.
    """
    df = pd.read_csv(config.UNIFIED_CSV)
    print(f"\n📊 Dataset: {len(df):,} images | "
          f"DR+: {df['binary_label'].sum():.0f} "
          f"({df['binary_label'].mean()*100:.1f}%)")
    df = remap_image_paths(df)

    train_df, temp_df = train_test_split(
        df, test_size=(1 - config.TRAIN_RATIO),
        stratify=df['binary_label'], random_state=42,
    )
    val_ratio_adj = config.VAL_RATIO / (config.VAL_RATIO + config.TEST_RATIO)
    val_df, test_df = train_test_split(
        temp_df, test_size=(1 - val_ratio_adj),
        stratify=temp_df['binary_label'], random_state=42,
    )
    print(f"  Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
    return train_df, val_df, test_df


# =============================================================================
# SECTION 5 — MODEL  (identical architecture to NB9)
# =============================================================================

class DRClassifier(nn.Module):
    def __init__(self, model_name=MODEL_NAME, pretrained=True, dropout=0.4):
        super().__init__()
        self.model_name = model_name
        timm_name = MODEL_REGISTRY[model_name]["timm_name"]

        self.backbone = timm.create_model(
            timm_name, pretrained=pretrained,
            num_classes=0, global_pool='avg',
        )

        # Detect REAL output dim via dummy forward
        # (timm.num_features unreliable for MobileNetV3 etc.)
        self.backbone.eval()
        with torch.no_grad():
            _out = self.backbone(torch.zeros(1, 3, 224, 224))
            feature_dim = int(_out.shape[-1])
        del _out
        print(f"  🔧 {model_name}: feature_dim={feature_dim} (detected via dummy forward)")

        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 1),
        )

        n_train = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  ✅ Trainable params: {n_train:,}")

    def forward(self, x):
        return self.classifier(self.backbone(x)).squeeze(-1)

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def freeze_bn(self):
        for m in self.modules():
            if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                m.eval()
                for p in m.parameters():
                    p.requires_grad = False

    def get_param_groups(self, bb_lr, head_lr):
        return [
            {'params': self.backbone.parameters(),   'lr': bb_lr},
            {'params': self.classifier.parameters(),  'lr': head_lr},
        ]


# =============================================================================
# SECTION 6 — EVALUATION
# =============================================================================

def evaluate(model, loader, device):
    # Unwrap DataParallel if present — avoids BN-replica CUBLAS failures
    m = model.module if isinstance(model, nn.DataParallel) else model
    m.eval()
    yt, yp = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            probs = torch.sigmoid(m(x)).cpu().numpy()
            yp.extend(probs.tolist())
            yt.extend(y.numpy().tolist())
    yt = np.array(yt)
    yp = np.array(yp)
    yh = (yp > 0.5).astype(int)
    try:
        auc = roc_auc_score(yt, yp)
    except ValueError:
        auc = 0.5
    return {
        'auc':       auc,
        'f1':        f1_score(yt, yh, zero_division=0),
        'precision': precision_score(yt, yh, zero_division=0),
        'recall':    recall_score(yt, yh, zero_division=0),
        'accuracy':  accuracy_score(yt, yh),
        'y_true': yt, 'y_prob': yp, 'y_pred': yh,
    }


# =============================================================================
# SECTION 7 — CENTRALIZED TRAINING
# =============================================================================

def train_centralized(train_df, val_loader, test_loader, seed=42):
    """
    Standard supervised training on full train_df.

    Schedule:
      Epochs 1–FREEZE_EPOCHS  : backbone frozen, head only
      Epochs FREEZE_EPOCHS+1… : full model + cosine LR decay
      Early stop               : patience=5 on val AUC
    """
    set_seed(seed)
    device = config.DEVICE
    t0     = time.time()

    # ── Model ─────────────────────────────────────────────────────────────
    # NOTE: DataParallel is intentionally NOT used here.
    # freeze_bn() sets BN layers to .eval() inside DataParallel replicas,
    # which causes CUBLAS_STATUS_EXECUTION_FAILED on the second GPU during
    # evaluation. Single-GPU training on T4 (16 GB) is sufficient for
    # centralized training at batch_size=128.
    model = DRClassifier(MODEL_NAME, pretrained=config.PRETRAINED,
                         dropout=config.DROPOUT).to(device)
    base  = model   # no DataParallel wrapper — base == model directly

    # ── DataLoader ─────────────────────────────────────────────────────────
    train_loader = DataLoader(
        DRDataset(train_df, get_transforms('train')),
        batch_size=config.BATCH_SIZE, shuffle=True,
        num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY,
        worker_init_fn=worker_init_fn,
        generator=torch.Generator().manual_seed(seed),
    )

    # ── Loss + optimizer ───────────────────────────────────────────────────
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([config.POS_WEIGHT], device=device)
    )
    optimizer = optim.AdamW(
        base.get_param_groups(config.BACKBONE_LR, config.HEAD_LR),
        weight_decay=config.WEIGHT_DECAY,
    )

    if config.USE_AMP:
        scaler = GradScaler('cuda') if _AMP_NEW else GradScaler()

    scheduler    = None
    history      = {'train_loss': [], 'val_auc': [], 'test_auc': []}
    best_val_auc = 0.0
    best_state   = None
    patience_cnt = 0
    best_epoch   = 0

    print(f"\n  🚀 Centralized [{MODEL_NAME}] seed={seed} | "
          f"bs={config.BATCH_SIZE} | freeze={config.FREEZE_EPOCHS}ep")

    pbar = tqdm(range(1, config.MAX_EPOCHS + 1),
                desc=f"  [{MODEL_NAME[:14]}] seed={seed}")

    for epoch in pbar:
        # ── Freeze / unfreeze ──────────────────────────────────────────
        if epoch <= config.FREEZE_EPOCHS:
            base.freeze_backbone()
        else:
            base.unfreeze_backbone()

        if config.FREEZE_BN:
            base.freeze_bn()

        # ── Cosine scheduler starts after freeze phase ─────────────────
        if epoch == config.FREEZE_EPOCHS + 1:
            T_max = config.MAX_EPOCHS - config.FREEZE_EPOCHS
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=T_max, eta_min=1e-7,
            )
        elif scheduler is not None and epoch > config.FREEZE_EPOCHS + 1:
            scheduler.step()

        # ── Train one epoch ────────────────────────────────────────────
        model.train()
        if config.FREEZE_BN:
            base.freeze_bn()

        epoch_loss = 0.0
        n_batches  = 0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            if config.USE_AMP:
                ctx = autocast('cuda') if _AMP_NEW else autocast()
                with ctx:
                    loss = criterion(model(x), y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), config.GRAD_CLIP
                )
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = criterion(model(x), y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), config.GRAD_CLIP
                )
                optimizer.step()

            epoch_loss += loss.item()
            n_batches  += 1

        avg_loss = epoch_loss / max(n_batches, 1)

        # ── Validate ───────────────────────────────────────────────────
        val_m  = evaluate(model, val_loader,  device)
        test_m = evaluate(model, test_loader, device)

        history['train_loss'].append(avg_loss)
        history['val_auc'].append(val_m['auc'])
        history['test_auc'].append(test_m['auc'])

        # ── Best model tracking ────────────────────────────────────────
        improved = val_m['auc'] > best_val_auc + config.MIN_DELTA
        if improved:
            best_val_auc = val_m['auc']
            best_state   = copy.deepcopy(base.state_dict())
            patience_cnt = 0
            best_epoch   = epoch
        else:
            patience_cnt += 1

        pbar.set_postfix({
            'loss':   f"{avg_loss:.4f}",
            'val':    f"{val_m['auc']:.4f}",
            'test':   f"{test_m['auc']:.4f}",
            'best':   f"{best_val_auc:.4f}@e{best_epoch}",
            'pat':    f"{patience_cnt}/{config.PATIENCE}",
        })

        # ── Early stopping ─────────────────────────────────────────────
        if patience_cnt >= config.PATIENCE and epoch > config.FREEZE_EPOCHS:
            print(f"\n  ⏹  Early stop at epoch {epoch} "
                  f"(best val AUC={best_val_auc:.4f} @ epoch {best_epoch})")
            break

    # ── Restore best checkpoint ────────────────────────────────────────
    if best_state is not None:
        base.load_state_dict(best_state)

    final   = evaluate(model, test_loader, device)
    elapsed = time.time() - t0

    print(f"\n  ✅ seed={seed} | AUC={final['auc']:.4f} | "
          f"F1={final['f1']:.4f} | Recall={final['recall']:.4f} | "
          f"Best epoch={best_epoch}/{epoch} | Time={elapsed/60:.1f}min")

    del model
    torch.cuda.empty_cache()
    gc.collect()

    return {
        'final':        final,
        'history':      history,
        'best_epoch':   best_epoch,
        'total_epochs': epoch,
        'elapsed_sec':  elapsed,
        'model_name':   MODEL_NAME,
        'seed':         seed,
    }


# =============================================================================
# SECTION 8 — AGGREGATE SEEDS + SAVE
# =============================================================================

def aggregate_and_save(seed_results):
    aucs  = [r['final']['auc']       for r in seed_results]
    f1s   = [r['final']['f1']        for r in seed_results]
    precs = [r['final']['precision'] for r in seed_results]
    recs  = [r['final']['recall']    for r in seed_results]
    accs  = [r['final']['accuracy']  for r in seed_results]
    eps   = [r['total_epochs']       for r in seed_results]

    auc_mean = float(np.mean(aucs))
    auc_std  = float(np.std(aucs))

    model_summary = {
        'model_name':  MODEL_NAME,
        'description': MODEL_REGISTRY[MODEL_NAME]['description'],
        'color':       MODEL_REGISTRY[MODEL_NAME]['color'],
        'auc_mean':    auc_mean,
        'auc_std':     auc_std,
        'auc_ci_lo':   auc_mean - 1.96 * auc_std,
        'auc_ci_hi':   auc_mean + 1.96 * auc_std,
        'f1_mean':     float(np.mean(f1s)),
        'prec_mean':   float(np.mean(precs)),
        'rec_mean':    float(np.mean(recs)),
        'acc_mean':    float(np.mean(accs)),
        'avg_epochs':  float(np.mean(eps)),
        'elapsed_min': sum(r['elapsed_sec'] for r in seed_results) / 60,
        'seed_results': seed_results,
    }

    # Save to shared dir so NB8 can load all models together
    save_path = Path(config.SAVE_DIR) / f"results_{MODEL_NAME}.pkl"
    with open(save_path, 'wb') as f:
        pickle.dump({MODEL_NAME: model_summary}, f,
                    protocol=pickle.HIGHEST_PROTOCOL)
    print(f"\n  💾 Saved → {save_path}")

    print(f"\n{'='*60}")
    print(f"📊 {MODEL_NAME} — FINAL SUMMARY")
    print(f"   AUC  : {auc_mean:.4f} ± {auc_std:.4f} "
          f"[{auc_mean - 1.96*auc_std:.4f} – {auc_mean + 1.96*auc_std:.4f}]")
    print(f"   F1   : {np.mean(f1s):.4f}  |  Recall: {np.mean(recs):.4f}  "
          f"|  Precision: {np.mean(precs):.4f}")
    print(f"   Epochs: {np.mean(eps):.1f} avg | "
          f"Time: {model_summary['elapsed_min']:.1f} min total")
    print(f"{'='*60}")

    return model_summary


# =============================================================================
# SECTION 9 — CONVERGENCE PLOT FOR THIS MODEL
# =============================================================================

def plot_single_model(model_summary):
    seed_results = model_summary['seed_results']
    color        = model_summary['color']

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(
        f"Centralized Training — {MODEL_NAME}\n{model_summary['description']}",
        fontsize=12, fontweight='bold',
    )

    # (a) Val AUC per epoch per seed
    ax = axes[0]
    for r in seed_results:
        epochs = range(1, len(r['history']['val_auc']) + 1)
        ax.plot(epochs, r['history']['val_auc'],
                color=color, alpha=0.5, linewidth=1.2,
                label=f"seed {r['seed']}")
    # Mean (pad shorter run)
    max_len = max(len(r['history']['val_auc']) for r in seed_results)
    padded  = [r['history']['val_auc'] +
               [r['history']['val_auc'][-1]] * (max_len - len(r['history']['val_auc']))
               for r in seed_results]
    mean_v = np.mean(padded, axis=0)
    ax.plot(range(1, max_len + 1), mean_v, color=color,
            linewidth=2.5, label='Mean', linestyle='-')
    ax.axvline(x=model_summary['avg_epochs'], color='gray',
               linestyle=':', alpha=0.6, label='Avg stop epoch')
    ax.set_title('(a) Val AUC per Epoch')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('AUC')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # (b) Test AUC per epoch per seed
    ax = axes[1]
    for r in seed_results:
        epochs = range(1, len(r['history']['test_auc']) + 1)
        ax.plot(epochs, r['history']['test_auc'],
                color=color, alpha=0.5, linewidth=1.2,
                label=f"seed {r['seed']}")
    padded_t = [r['history']['test_auc'] +
                [r['history']['test_auc'][-1]] * (max_len - len(r['history']['test_auc']))
                for r in seed_results]
    mean_t = np.mean(padded_t, axis=0)
    ax.plot(range(1, max_len + 1), mean_t, color=color,
            linewidth=2.5, label='Mean')
    # Mark best final AUC
    ax.axhline(y=model_summary['auc_mean'], color='black',
               linestyle='--', alpha=0.5,
               label=f"Final: {model_summary['auc_mean']:.4f}")
    ax.set_title('(b) Test AUC per Epoch')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('AUC')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # (c) Training loss
    ax = axes[2]
    for r in seed_results:
        ax.plot(range(1, len(r['history']['train_loss']) + 1),
                r['history']['train_loss'],
                color=color, alpha=0.6, linewidth=1.5,
                label=f"seed {r['seed']}")
    ax.set_title('(c) Training Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE Loss')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plot_path = MODEL_SAVE_DIR / f"{MODEL_NAME}_centralized_curves.png"
    fig.savefig(plot_path, dpi=130, bbox_inches='tight')
    plt.close(fig)
    print(f"  📈 Plot saved → {plot_path}")


# =============================================================================
# SECTION 10 — ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    print("\n" + "="*60)
    print(f"🚀 NB10 — Centralized Single-Model Run")
    print(f"   Model: {MODEL_NAME}")
    print(f"   {MODEL_REGISTRY[MODEL_NAME]['description']}")
    print("="*60)

    # ── Load data (stratified split identical to NB9) ─────────────────────
    print("\n📂 Loading dataset...")
    train_df, val_df, test_df = load_and_split_data()

    val_loader = DataLoader(
        DRDataset(val_df,  get_transforms('val')),
        batch_size=256, shuffle=False,
        num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY,
    )
    test_loader = DataLoader(
        DRDataset(test_df, get_transforms('val')),
        batch_size=256, shuffle=False,
        num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY,
    )

    # ── Run both seeds sequentially ───────────────────────────────────────
    seed_results = []
    for seed in config.SEEDS:
        print(f"\n{'─'*40}")
        print(f"  Seed {seed}")
        print(f"{'─'*40}")
        result = train_centralized(
            train_df, val_loader, test_loader, seed=seed
        )
        seed_results.append(result)
        gc.collect()
        torch.cuda.empty_cache()

    # ── Aggregate, save, and plot ─────────────────────────────────────────
    model_summary = aggregate_and_save(seed_results)
    plot_single_model(model_summary)

    print(f"\n✅ DONE — {MODEL_NAME}")
    print(f"   Results pkl → "
          f"{Path(config.SAVE_DIR) / f'results_{MODEL_NAME}.pkl'}")
    print(f"   Curves plot → "
          f"{MODEL_SAVE_DIR / f'{MODEL_NAME}_centralized_curves.png'}")
    print(f"\n   To run next model: change MODEL_NAME at the top of this notebook")
    print(f"\n   Federation cost (after NB9 + NB10 both done):")
    print(f"   Δ_fed = Centralized AUC − FedAdapt-EF AUC")

timm 1.0.24 already installed
✅ All imports done
PyTorch: 2.9.0+cu126 | timm: 1.0.24
CUDA: True
  GPU 0: Tesla T4
  GPU 1: Tesla T4

📋 Running: densenet121
   DenseNet-121 (8M) — Medical imaging standard
   batch_size=96

✅ Config ready
   Model:      densenet121
   Seeds:      [42, 123]
   Max epochs: 30 (early stop patience=5)
   Batch size: 96
   Save dir:   /kaggle/working/results/centralized_multimodel/densenet121

🚀 NB10 — Centralized Single-Model Run
   Model: densenet121
   DenseNet-121 (8M) — Medical imaging standard

📂 Loading dataset...

📊 Dataset: 16,652 images | DR+: 8422 (50.6%)
  ⚠️  Broken paths (12/20 exist) — scanning /kaggle/input/ ...
    [competitions]: 5,590 images indexed
    [datasets]: 14,094 images indexed
  📊 Total indexed: 19,684 images
  ✅ After remap: 20/20 sampled paths valid
  Train: 11,656 | Val: 2,498 | Test: 2,498

────────────────────────────────────────
  Seed 42
────────────────────────────────────────


model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

  🔧 densenet121: feature_dim=1024 (detected via dummy forward)
  ✅ Trainable params: 7,479,169

  🚀 Centralized [densenet121] seed=42 | bs=96 | freeze=3ep


  [densenet121] seed=42:   0%|          | 0/30 [00:00<?, ?it/s]


  ⏹  Early stop at epoch 21 (best val AUC=0.9694 @ epoch 16)

  ✅ seed=42 | AUC=0.9690 | F1=0.9078 | Recall=0.9121 | Best epoch=16/21 | Time=104.6min

────────────────────────────────────────
  Seed 123
────────────────────────────────────────
  🔧 densenet121: feature_dim=1024 (detected via dummy forward)
  ✅ Trainable params: 7,479,169

  🚀 Centralized [densenet121] seed=123 | bs=96 | freeze=3ep


  [densenet121] seed=123:   0%|          | 0/30 [00:00<?, ?it/s]


  ⏹  Early stop at epoch 19 (best val AUC=0.9693 @ epoch 14)

  ✅ seed=123 | AUC=0.9686 | F1=0.9106 | Recall=0.9034 | Best epoch=14/19 | Time=93.7min

  💾 Saved → /kaggle/working/results/centralized_multimodel/results_densenet121.pkl

📊 densenet121 — FINAL SUMMARY
   AUC  : 0.9688 ± 0.0002 [0.9684 – 0.9693]
   F1   : 0.9092  |  Recall: 0.9078  |  Precision: 0.9107
   Epochs: 20.0 avg | Time: 198.3 min total
  📈 Plot saved → /kaggle/working/results/centralized_multimodel/densenet121/densenet121_centralized_curves.png

✅ DONE — densenet121
   Results pkl → /kaggle/working/results/centralized_multimodel/results_densenet121.pkl
   Curves plot → /kaggle/working/results/centralized_multimodel/densenet121/densenet121_centralized_curves.png

   To run next model: change MODEL_NAME at the top of this notebook

   Federation cost (after NB9 + NB10 both done):
   Δ_fed = Centralized AUC − FedAdapt-EF AUC
